# 230 — Blob clustering

![image.png](attachment:2ae06b53-2ef7-4c2c-81f2-c99f94beed92.png)


Blob feature extraction → score gating → feature weighting → KMeans + HC clustering.

**Outputs** → `outputs/230_blob_clustering_runs/<run_id>/`

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

sys.path.insert(0, str(Path('..').resolve()))

from functions import lf_blob_clustering_config as cfg
from functions.lf_blob_clustering import (
    s20_build_valley_features,
    s30_gate_by_blob_score,
    q31_plot_score_histograms,
    q32_plot_score_gating_grid,
    q33_plot_valley_overlay_grid,
)
from functions.lf_blob_metrics import s22_build_blob_feature_matrix, s23_init_blob_metric
from functions.lf_clustering import s10_load_ersps, s71_save_gated_blob_run
from functions.lf_clustering_methods import (
    s50_run_kmeans,
    s51_run_hc,
    s52_cut_hc,
    s53_compute_medoids,
    s55_build_df_with_clusters,
    s71_save_run,
    q50_plot_silhouette_curve,
    q51_plot_dendrogram,
    q52_plot_prototype_grid,
)

# ── output root ──────────────────────────────────────────────
BASE_RUNS_DIR = Path('outputs/230_blob_clustering_runs')
BASE_RUNS_DIR.mkdir(parents=True, exist_ok=True)

## Config

In [ ]:
# ── data input ───────────────────────────────────────────────
INPUT_DIR = Path('../01_FBM_Analysis/outputs/04_ersp_LM_RAWONLY')
TASK      = 'LM'
CONDITIONS = ('audio', 'picture', 'reading')
N_FREQ, N_TIME = 129, 300

# ── segmentation ─────────────────────────────────────────────
VALLEY_PARAMS = cfg.VALLEY_PARAMS

# ── score gating ─────────────────────────────────────────────
BLOB_SCORE_QUANTILE  = cfg.BLOB_SCORE_QUANTILE
MANUAL_SCORE_THR     = None   # set a float to override quantile

# ── feature weighting ────────────────────────────────────────
FEATURE_TYPE_WEIGHTS = cfg.FEATURE_TYPE_WEIGHTS

# ── clustering ───────────────────────────────────────────────
KMEANS_K_RANGE = cfg.KMEANS_K_RANGE
HC_METHOD      = 'average'      # 'average', 'complete', 'ward'
HC_N_CLUSTERS  = 20             # cut HC at this K for comparison
RANDOM_STATE   = cfg.RANDOM_STATE

print('VALLEY_PARAMS:', VALLEY_PARAMS)
print('BLOB_SCORE_QUANTILE:', BLOB_SCORE_QUANTILE)
print('KMEANS_K_RANGE:', KMEANS_K_RANGE)

## 1 — Load ERSPs

In [ ]:
df_meta, ersp_list = s10_load_ersps(
    input_dir=INPUT_DIR,
    task=TASK,
    allowed_conditions=CONDITIONS,
    n_freq=N_FREQ,
    n_time=N_TIME,
)
print(df_meta.shape, len(ersp_list))
df_meta.head()

## 2 — Valley blob segmentation + feature extraction

In [ ]:
X_blob, max_scores, blobs_per_sample = s22_build_blob_feature_matrix(
    ersp_list=ersp_list,
    max_blobs=int(VALLEY_PARAMS['max_blobs']),
    thr_pos=float(VALLEY_PARAMS['thr_pos']),
    thr_neg=float(VALLEY_PARAMS['thr_neg']),
    delta_valley=float(VALLEY_PARAMS['delta_valley']),
    min_mean_pos=float(VALLEY_PARAMS['min_mean_pos']),
    max_mean_neg=float(VALLEY_PARAMS['max_mean_neg']),
    sign_mode=str(VALLEY_PARAMS['sign_mode']),
)
print('X_blob:', X_blob.shape)
print('max_scores:', max_scores.shape, '  min/max:', max_scores.min(), max_scores.max())

## 3 — Score gating

In [ ]:
q31_plot_score_histograms(
    max_scores=max_scores,
    score_thresh=float(np.quantile(max_scores, BLOB_SCORE_QUANTILE)),
    quantile=BLOB_SCORE_QUANTILE,
)

In [ ]:
(
    df_keep, ersp_keep, X_blob_keep,
    keep_idx, score_thr, keep_mask, drop_idx
) = s30_gate_by_blob_score(
    df_meta=df_meta.copy(),
    ersp_list=ersp_list,
    X_blob=X_blob,
    max_scores=max_scores,
    quantile=BLOB_SCORE_QUANTILE,
    manual_threshold=MANUAL_SCORE_THR,
)
print('Kept:', len(df_keep), '  Dropped:', len(drop_idx))

In [ ]:
q32_plot_score_gating_grid(
    ersp_list_raw=ersp_list,
    df_meta_raw=df_meta,
    keep_idx=keep_idx,
    drop_idx=drop_idx,
    score_thresh=score_thr,
)

## 4 — Feature weighting

In [ ]:
s23_init_blob_metric(
    max_blobs=int(VALLEY_PARAMS['max_blobs']),
    features_per_blob=8,
    feature_type_weights=FEATURE_TYPE_WEIGHTS,
)

# Build weight vector and apply
from functions.lf_blob_metrics import BLOB_WEIGHT_VEC
Xw = X_blob_keep * BLOB_WEIGHT_VEC
print('Xw:', Xw.shape)

## 5 — KMeans clustering

In [ ]:
labels_km, best_k, sil_by_k, labels_by_k = s50_run_kmeans(
    Xw=Xw,
    k_range=KMEANS_K_RANGE,
    random_state=RANDOM_STATE,
)

In [ ]:
q50_plot_silhouette_curve(sil_by_k, best_k)

## 6 — Hierarchical clustering

In [ ]:
Z, D = s51_run_hc(Xw, method=HC_METHOD)

In [ ]:
# Add leaf labels for dendrogram
df_keep['leaf_label'] = (
    df_keep['patient_id'].astype(str) + '_' +
    df_keep['electrode'].astype(str) + '_' +
    df_keep['condition'].astype(str)
)

q51_plot_dendrogram(
    Z=Z,
    df_leaf=df_keep,
    strip_col='condition',
    title=f'HC blob features — method={HC_METHOD}',
)

In [ ]:
labels_hc = s52_cut_hc(Z, n_clusters=HC_N_CLUSTERS)
print('HC labels — unique clusters:', len(np.unique(labels_hc)))

## 7 — Medoids

In [ ]:
medoids_km = s53_compute_medoids(D, labels_km)
medoids_hc = s53_compute_medoids(D, labels_hc)
print('KMeans medoids:', medoids_km)
print('HC medoids:', medoids_hc)

## 8 — Save run

In [ ]:
# Save gated blob run first (X_blob_keep, df_keep, ersp_keep, scores)
run_dir = s71_save_gated_blob_run(
    base_runs_dir=BASE_RUNS_DIR,
    X_blob_keep=X_blob_keep,
    df_keep=df_keep,
    keep_idx=keep_idx,
    drop_idx=drop_idx,
    score_thr=score_thr,
    blob_score_quantile=BLOB_SCORE_QUANTILE,
    valley_params=VALLEY_PARAMS,
    ersp_keep=ersp_keep,
    max_scores=max_scores,
)
print('Run dir:', run_dir)

In [ ]:
# KMeans cluster assignments
param_tag_km = f'k{best_k}_q{BLOB_SCORE_QUANTILE:.2f}'.replace('.', 'p')
df_km = s55_build_df_with_clusters(
    df_keep=df_keep,
    labels=labels_km,
    algo_tag='kmeans_blob',
    param_tag=param_tag_km,
    keep_idx=keep_idx,
)

s71_save_run(
    run_dir=run_dir,
    df_keep_with_clusters=df_km,
    labels=labels_km,
    algo_tag='kmeans_blob',
    param_tag=param_tag_km,
    sil_by_k=sil_by_k,
    labels_by_k=labels_by_k,
    extra_meta={'valley_params': VALLEY_PARAMS, 'feature_weights': FEATURE_TYPE_WEIGHTS},
)

In [ ]:
# HC cluster assignments
param_tag_hc = f'k{HC_N_CLUSTERS}_{HC_METHOD}'
df_hc = s55_build_df_with_clusters(
    df_keep=df_keep,
    labels=labels_hc,
    algo_tag='hc_blob',
    param_tag=param_tag_hc,
    keep_idx=keep_idx,
)

s71_save_run(
    run_dir=run_dir,
    df_keep_with_clusters=df_hc,
    labels=labels_hc,
    algo_tag='hc_blob',
    param_tag=param_tag_hc,
    Z=Z,
    D=D,
    extra_meta={'valley_params': VALLEY_PARAMS, 'hc_method': HC_METHOD, 'hc_n_clusters': HC_N_CLUSTERS},
)
print('Done — run saved to:', run_dir)